# Commons Simulation - ANOVA Analysis

Bu notebook, `output/model_metrics_104x25.csv` ve `output/scenario_matrix_104.csv` dosyalarini kullanarak
ana metrikler icin ANOVA + eta-squared + Tukey HSD analizi uretir ve akademik rapor ozeti hazirlar.

In [ ]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

MODEL_PATH = 'output/model_metrics_104x25.csv'
SCENARIO_PATH = 'output/scenario_matrix_104.csv'

assert os.path.exists(MODEL_PATH), f'Missing file: {MODEL_PATH}'
assert os.path.exists(SCENARIO_PATH), f'Missing file: {SCENARIO_PATH}'

In [ ]:
df = pd.read_csv(MODEL_PATH)
scenario_df = pd.read_csv(SCENARIO_PATH)
print('model_df shape:', df.shape)
print('scenario_df shape:', scenario_df.shape)
display(df.head(3))
display(scenario_df.head(3))

In [ ]:
# Her run icin final tick satirini al
final_df = (
    df.sort_values(['scenario_id', 'repeat_id', 'tick'])
    .groupby(['scenario_id', 'repeat_id'], as_index=False)
    .tail(1)
    .copy()
)
print('final_df shape:', final_df.shape)
display(final_df[['scenario_id','repeat_id','system_type','config_name','gini_coefficient','mean_trust','resource_utilization','free_rider_ratio']].head())

In [ ]:
metrics = ['gini_coefficient', 'mean_trust', 'resource_utilization', 'free_rider_ratio']
anova_tables = {}
tukey_results = {}

for metric in metrics:
    model = ols(
        f'{metric} ~ C(system_type) + C(cohesion) + N_people + N_resources',
        data=final_df,
    ).fit()

    try:
        anova = sm.stats.anova_lm(model, typ=2)
    except Exception:
        anova = sm.stats.anova_lm(model, typ=1)

    ss_total = anova['sum_sq'].sum()
    anova = anova.copy()
    anova['eta_sq'] = anova['sum_sq'] / ss_total
    anova_tables[metric] = anova

    if final_df['system_type'].nunique() >= 2:
        tukey = pairwise_tukeyhsd(
            endog=final_df[metric],
            groups=final_df['system_type'],
            alpha=0.05,
        )
    else:
        tukey = 'Tukey skipped: only one system_type present in data.'
    tukey_results[metric] = tukey

print('ANOVA and Tukey completed for all metrics.')

In [ ]:
# ANOVA tablolarini goster
for metric in metrics:
    print('\n' + '='*80)
    print(f'ANOVA: {metric}')
    display(anova_tables[metric])

In [ ]:
# Tukey HSD ozetleri
for metric in metrics:
    print('\n' + '='*80)
    print(f'Tukey HSD: {metric}')
    print(tukey_results[metric])

In [ ]:
# Hipotez bazli kisa yorumlarin otomatik iskeleti
summary_rows = []
for metric in metrics:
    table = anova_tables[metric]
    p_system = table.loc['C(system_type)', 'PR(>F)'] if 'C(system_type)' in table.index else None
    eta_system = table.loc['C(system_type)', 'eta_sq'] if 'C(system_type)' in table.index else None
    summary_rows.append({'metric': metric, 'p_system_type': p_system, 'eta_sq_system_type': eta_system})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
# Akademik rapora yapistirmalik mini ozet metni
for _, row in summary_df.iterrows():
    metric = row['metric']
    pval = row['p_system_type']
    eta = row['eta_sq_system_type']
    sig = 'anlamli' if pval is not None and pval < 0.05 else 'anlamli degil'
    p_txt = f'{pval:.4g}' if pval is not None else 'NA'
    eta_txt = f'{eta:.4f}' if eta is not None else 'NA'
    print(f'- {metric}: system_type etkisi {sig} (p={p_txt}, eta^2={eta_txt})')